# 状态

## 状态定义类型

1. 推荐状态继承`TypedDict`

## 节点接收的state

state参数它是一份快照（或者说只读视图）

## 节点返回值

1. 默认采用增量更新
2. 使用了Annotated定义了 `reduce`，函数签名：`(Value, Value) -> Value`，接收当前值和更新值，返回合并后的新值。
    1. `langgraph.graph.message.add_messages`是 LangGraph 中专用于合并消息列表的 Reducer 函数，常用于维护对话历史类的状态字段

对于节点返回的字段，LangGraph 会根据该字段是否配置了 `Reducer` 来决定如何合并更新值。

- 如果字段配置了 `Reducer`，则使用对应的 `Reducer` 函数将旧值和新值合并；
- 如果字段没有配置 `Reducer`，则按照默认规则使用节点返回的新值覆盖原值

In [1]:
import os

import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

In [2]:
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState


class Demo1State(MessagesState):
    username: str
    output: str


def node_1(state: Demo1State):
    return {"messages": [HumanMessage(content=f"你好我是沉稳的{state['username']}")]}


def llm_node(state: Demo1State):
    ai_msg = model.invoke(state["messages"])
    ai_msg.pretty_print()
    return {"messages": [ai_msg], "output": ai_msg.content}

In [3]:
from langgraph.constants import END, START
from langgraph.graph import StateGraph

graph_builder = StateGraph(state_schema=Demo1State)
graph_builder.add_node(node_1).add_node(llm_node)
graph_builder.add_edge(START, "node_1").add_edge("node_1", "llm_node").add_edge(
    "llm_node", END
)

graph = graph_builder.compile()

In [4]:
result = graph.invoke({"username": "鲨鱼不吃辣椒"})

================================== Ai Message ==================================

你好，沉稳的鲨鱼不吃辣椒！很高兴认识你 🦈🌊  
不吃辣椒确实很符合“沉稳”人设。有什么想聊的或需要帮忙的吗？


In [5]:
from pprint import pprint

pprint(result)

{'messages': [HumanMessage(content='你好我是沉稳的鲨鱼不吃辣椒', additional_kwargs={}, response_metadata={}, id='3359bb0c-9c7c-40ad-9314-229e88ccc899'),
              AIMessage(content='你好，沉稳的鲨鱼不吃辣椒！很高兴认识你 🦈🌊  \n不吃辣椒确实很符合“沉稳”人设。有什么想聊的或需要帮忙的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 38, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 89, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 38}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V4.1-Flash', 'system_fingerprint': '', 'id': '01a1590510f5b9a641e85b44b7d599c6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15904-e63f-7953-9677-5d386faae50a-0', tool_calls=[]

# 私有状态与限定输入输出

四类状态的定义和使用,其中私有状态可以理解为各个节点的隔离

In [6]:
from typing import TypedDict


class GlobalState(TypedDict):
    username: str
    nickname: str
    output: str


class InputState(TypedDict):
    username: str


class OutputState(TypedDict):
    output: str


class PrivateState(TypedDict):
    greeting: str

In [7]:
def node_1(state: InputState):
    return {"nickname": "沉稳の" + state["username"]}


def node_2(state: GlobalState):
    """从全局状态读取数据，写入私有状态"""
    return {"greeting": state["nickname"] + "在深圳Happy Coding"}


def node_3(state: PrivateState):
    ai_msg = model.invoke(state["greeting"])
    return {"output": ai_msg.content}

In [8]:
graph_builder = StateGraph(
    state_schema=GlobalState, input_schema=InputState, output_schema=OutputState
)

graph_builder.add_node(node_1).add_node(node_2).add_node(node_3)

graph_builder.add_edge(START, "node_1").add_edge("node_1", "node_2").add_edge(
    "node_2", "node_3"
).add_edge("node_3", END)

graph = graph_builder.compile()

In [9]:
result = graph.invoke({"username": "胖卡"})

In [10]:
type(result)

dict

In [11]:
print(result["output"])

收到～沉稳の胖卡在深圳 Happy Coding 🐱💻  
祝 coding 顺利！有什么需要我帮忙的吗？比如代码、报错、项目思路都可以发来。


# 状态存储Checkpoint

将第一次调用的结果，存储起来，然后在第二次调用时，**从存储的结果当中取出状态，而不是让LangGraph去初始化空状态**。
由于同一个agent会存在多用户或者单用户多会话等场景，要实现状态存储，并且后续能够找到正确的状态，就需要在状态当中做一个id映射，写入时指定一个id，读取时读取指定id的状态。

id隔离，这个id在langgraph当中就称之为thread_id。

图调用时，传递thread_id：保证上下文空间，就传入相同的thread_id，需要开辟新的上下文空间，就传入新的thread_id。

## 实现方案

检查点后端必须基于某个检查点实现，**常见**检查点实现和后端的对应关系如下

| 类名                | 包名                                | 检查点后端              |
| ------------------- | ----------------------------------- | ----------------------- |
| **`InMemorySaver`** | **`langgraph-checkpoint`**          | 内存（**`In-memory`**） |
| **`SqliteSaver`**   | **`langgraph-checkpoint-sqlite`**   | **`SQLite`**            |
| **`PostgresSaver`** | **`langgraph-checkpoint-postgres`** | **`PostgreSQL`**        |
| **`MongoDBSaver`**  | **`langgraph-checkpoint-mongodb`**  | **`MongoDB`**           |
| **`RedisSaver`**    | **`langgraph-checkpoint-redis`**    | **`Redis`**             |

在这些实现中，**`InMemorySaver`** 更适合学习、调试和本地实验；**`SQLite`** 适合轻量级本地持久化；**`PostgreSQL`**、**`MongoDB`**、**`Redis`** 等数据库后端更适合需要跨进程、跨服务保存状态的场景。

## SqliteSaver

1. 记忆： “接着上一轮对话”
2. 恢复：“续传没做完的任务”


In [ ]:
# sqlite3是python自动的标准库
import sqlite3

from langgraph.checkpoint.sqlite import SqliteSaver

print("sqlite3:", sqlite3.sqlite_version)

sqlite3: 3.50.4


In [13]:
import os

os.makedirs("./sqlite_data", exist_ok=True)

conn = sqlite3.connect(
    database="./sqlite_data/langgraph_sqlite", check_same_thread=False
)

checkpointer = SqliteSaver(conn)

### 记忆

- 第一次 invoke，图跑到了 END，任务已经完美结束了。
- 第二次 invoke，传了一个新的字典`（{"messages": [HumanMessage("...")]}）`，意思是：“上次聊完了，现在我们开启新一轮对话。”
- 之所以问“第一印象”，AI 还能答上来，是因为 checkpointer 会在新一轮对话开始时，自动把之前的历史消息（state）加载出来，并把你的新消息拼进去。

核心词：多轮记忆（Memory / Session）。

In [14]:
class MemoryState(MessagesState):
    output: str


def llm_node(state: MemoryState):
    ai_msg = model.invoke(state["messages"])
    return {"messages": [ai_msg]}


def node_out(state: MemoryState):
    return {"output": state["messages"][-1].content}

In [15]:
graph_builder = StateGraph(state_schema=MemoryState)

graph_builder.add_node(llm_node).add_node(node_out)
graph_builder.add_edge(START, llm_node.__name__)
graph_builder.add_edge(llm_node.__name__, node_out.__name__)
graph_builder.add_edge(node_out.__name__, END)

graph = graph_builder.compile(checkpointer=checkpointer)

In [16]:
print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| llm_node |   
+----------+   
      *        
      *        
      *        
+----------+   
| node_out |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [17]:
import time

config = {"configurable": {"thread_id": str(time.time_ns())}}

In [18]:
config

{'configurable': {'thread_id': '1792494864726275800'}}

In [19]:
graph.invoke(
    {"messages": [HumanMessage("Hi,我叫鲨鱼，目前在深圳，age 18")]}, config=config
)
graph.invoke({"messages": [HumanMessage("我正在学习Langgraph")]}, config=config)
result = graph.invoke({"messages": [HumanMessage("我给你的第一印象")]}, config=config)


In [20]:
from IPython.display import Markdown, display

display(Markdown(result["output"]))

基于目前这几句话，我的第一印象是：

1. **年轻，但方向感不弱**  
   18 岁在深圳，已经在学 LangGraph，说明你不是随便玩玩，而是在碰比较前沿的 LLM 应用编排 / Agent 方向。

2. **偏实践、爱折腾**  
   LangGraph 不是最入门的东西，愿意学它，感觉你更想“自己搭东西”，而不只是用现成聊天工具。

3. **说话直接、简洁**  
   你的自我介绍像报参数：名字、地点、年龄。很利落，有点工程师式表达，不喜欢废话。

4. **有记忆点**  
   叫“鲨鱼”本身就挺酷，第一反应是敏锐、速度快、不太安分——挺适合深圳这种节奏。

5. **可能常看英文技术资料**  
   你写“age 18”，中英混用，感觉你平时接触英文文档、技术社区应该不少。

一句话总结：**一个在深圳、18 岁、对 AI Agent 有热情、说话直接、愿意动手的年轻人。像一条刚下水但游得很快的鲨鱼。🦈**

准不准？如果不准，你可以纠正我。

In [21]:
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi,我叫鲨鱼，目前在深圳，age 18
================================== Ai Message ==================================

你好，鲨鱼！很高兴认识你。18岁在深圳，正是充满可能性的年纪呀。有什么想聊的、想了解的，或者需要我帮忙的吗？
================================ Human Message =================================

我正在学习Langgraph
================================== Ai Message ==================================

鲨鱼你好，18岁在深圳学 LangGraph，很酷！LangGraph 可以理解成：**用“图/状态机”的方式编排 LLM 应用**。它比普通 Chain 更适合做：带记忆的 Agent、循环推理、工具调用、多 Agent 协作、人在回路。

核心概念先抓住这几个：

- **State**：全局状态，通常用 `TypedDict` 定义
- **Node**：一个处理函数，接收 state，返回要更新的字段
- **Edge**：节点之间怎么走，有普通边和条件边
- **START / END**：图的入口和出口
- **Reducer**：状态字段怎么合并，比如 `add_messages`
- **Checkpointer**：持久化记忆，配合 `thread_id`
- **interrupt**：暂停图，等人类审批/输入

一个最小可跑例子，Python 版：

```python
# pip install langgraph langchain-openai
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_message

### 故障恢复

- 任务本来应该一口气跑完（Node1 -> Node2 -> Node3）。
- 但是 Node2 挂了，任务**中断**了。
- 传 `None`，是为了把这个**没跑完的残局**捡起来，继续跑完。等跑到了 END，这个任务就彻底结束了。

核心词：断点恢复（Resume）。

1. invoke时传递None作为初始参数；
2. 传入相同的thread_id。

In [22]:
config = {"configurable": {"thread_id": str(time.time_ns())}}

In [23]:
config

{'configurable': {'thread_id': '1792494913856302300'}}

In [24]:
from typing import Literal


class DeviceState(TypedDict):
    status: Literal["online", "offline"]
    key1: str
    key2: str
    key3: str


def node_1(state: DeviceState):
    """是一个非常耗时的操作"""
    print(f"node1耗时操作 初始状态为 {state}")
    return {"key1": "value1"}


def node_2(state: DeviceState):
    if state["status"] == "offline":
        raise Exception("设备已经不工作")  # noqa: TRY002
    print("node2 处理success")
    return {"key2": "value2"}


def node_3(state: DeviceState):
    print("node3 处理success")
    return {"key3": "value3"}

In [25]:
graph_builder = StateGraph(state_schema=DeviceState)

graph_builder.add_node(node_1)
graph_builder.add_node(node_2)
graph_builder.add_node(node_3)

graph_builder.add_edge(START, "node_1")
graph_builder.add_edge("node_1", "node_2")
graph_builder.add_edge("node_2", "node_3")
graph_builder.add_edge("node_3", END)
graph = graph_builder.compile(checkpointer=checkpointer)

In [26]:
# 模拟故障
try:
    graph.invoke({"status": "offline"}, config=config)
except Exception as e:  # noqa: BLE001
    print(e)

node1耗时操作 初始状态为 {'status': 'offline'}
设备已经不工作


In [27]:
graph.get_state(config=config)

StateSnapshot(values={'status': 'offline', 'key1': 'value1'}, next=('node_2',), config={'configurable': {'thread_id': '1792494913856302300', 'checkpoint_ns': '', 'checkpoint_id': '1f1cc778-5954-613c-8001-33da78e785d7'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-10-20T11:15:13.952154+00:00', parent_config={'configurable': {'thread_id': '1792494913856302300', 'checkpoint_ns': '', 'checkpoint_id': '1f1cc778-594b-65a8-8000-44f83f8646bd'}}, tasks=(PregelTask(id='a1fb76b9-b0b7-1f0b-8c07-7a2ee75984ce', name='node_2', path=('__pregel_pull', 'node_2'), error="Exception('设备已经不工作')", interrupts=(), state=None, result=None),), interrupts=())

模拟设备上线

In [28]:
graph.update_state(config=config, values={"status": "online"})
graph.get_state(config=config)

StateSnapshot(values={'status': 'online', 'key1': 'value1'}, next=('node_2',), config={'configurable': {'thread_id': '1792494913856302300', 'checkpoint_ns': '', 'checkpoint_id': '1f1cc778-5eef-695b-8002-7e49cd8ad424'}}, metadata={'source': 'update', 'step': 2, 'parents': {}}, created_at='2026-10-20T11:15:14.540147+00:00', parent_config={'configurable': {'thread_id': '1792494913856302300', 'checkpoint_ns': '', 'checkpoint_id': '1f1cc778-5954-613c-8001-33da78e785d7'}}, tasks=(PregelTask(id='2e0ba8ff-0de7-e010-8d2d-33089ae573ab', name='node_2', path=('__pregel_pull', 'node_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [29]:
graph.invoke(None, config=config)

node2 处理success
node3 处理success


{'status': 'online', 'key1': 'value1', 'key2': 'value2', 'key3': 'value3'}

上面的结果，很明显没有再处理node1

关于传入None的理解：没有新输入了（None），请你去数据库（checkpointer）里查一下这个 config（也就是这个 thread_id）对应的记录，看看它上次停在哪了，接着往下跑就行